In [1]:
import os
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "nn").exists() else Path.cwd().parent
os.chdir(ROOT)

In [2]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
from nn import *
from nn.activations import DERIVATIVES

# Gradient checking

Backprop gives a slope for every weight from formulas. Gradient checking measures the same slope by brute force: nudge one weight up and down by a tiny `eps`, rerun the forward pass, and see how much the loss moved.

$$\frac{\partial L}{\partial w} \approx \frac{L(w + \epsilon) - L(w - \epsilon)}{2\epsilon}$$

The two are compared with relative error, $|a - n| / (|a| + |n|)$, because gradients in different layers can differ in size by orders of magnitude.

- around `1e-7` or smaller: backprop is correct
- around `1e-4`: suspicious
- `1e-2` or larger: a derivative is wrong

In [3]:
# 3 inputs -> 4 tanh -> 1 sigmoid, binary cross-entropy
rng = np.random.default_rng(0)
net = stackLayers([
    multipleNeuron(n=4, n_inputs=3, activation=tanh, W=rng.normal(size=(4, 3)), b=rng.normal(0, 0.5, 4)),
    multipleNeuron(n=1, n_inputs=4, activation=sigmoid, W=rng.normal(size=(1, 4)), b=rng.normal(0, 0.5, 1)),
])
print(net)

rows = gradient_check(net, [1.0, 2.0, -1.0], [1.0], binary_cross_entropy)
print_gradient_check(rows)

stackLayers(3 -> 4 tanh -> 1 sigmoid)
param            analytic      numeric   rel error
L0 n0 w0       0.02028859   0.02028859    1.73e-10
L0 n0 w1       0.04057717   0.04057717    1.01e-10
L0 n0 w2      -0.02028859  -0.02028859    1.73e-10
L0 n0 b        0.02028859   0.02028859    1.73e-10
L0 n1 w0       0.03015311   0.03015311    1.18e-10
L0 n1 w1       0.06030621   0.06030621    7.16e-11
L0 n1 w2      -0.03015311  -0.03015311    1.18e-10
L0 n1 b        0.03015311   0.03015311    1.18e-10
L0 n2 w0      -0.00109817  -0.00109817    2.20e-09
L0 n2 w1      -0.00219633  -0.00219633    2.20e-09
L0 n2 w2       0.00109817   0.00109817    2.20e-09
L0 n2 b       -0.00109817  -0.00109817    2.20e-09
L0 n3 w0      -0.00569136  -0.00569136    5.46e-10
L0 n3 w1      -0.01138271  -0.01138271    1.86e-10
L0 n3 w2       0.00569136   0.00569136    5.46e-10
L0 n3 b       -0.00569136  -0.00569136    5.46e-10
L1 n0 w0       0.45237212   0.45237212    4.32e-12
L1 n0 w1       0.42103985   0.42103985    6.

## Breaking it on purpose

A check that always passes proves nothing. Swap in a wrong tanh slope, `1 - tanh(z)` instead of `1 - tanh(z)**2`, and the first layer's rows should light up.

In [4]:
original = DERIVATIVES[tanh]
DERIVATIVES[tanh] = lambda z: 1 - np.tanh(z)
try:
    broken = gradient_check(net, [1.0, 2.0, -1.0], [1.0], binary_cross_entropy)
finally:
    DERIVATIVES[tanh] = original

print_gradient_check(broken)

param            analytic      numeric   rel error
L0 n0 w0       0.50276836   0.02028859    9.22e-01
L0 n0 w1       1.00553673   0.04057717    9.22e-01
L0 n0 w2      -0.50276836  -0.02028859    9.22e-01
L0 n0 b        0.50276836   0.02028859    9.22e-01
L0 n1 w0       0.28227717   0.03015311    8.07e-01
L0 n1 w1       0.56455435   0.06030621    8.07e-01
L0 n1 w2      -0.28227717  -0.03015311    8.07e-01
L0 n1 b        0.28227717   0.03015311    8.07e-01
L0 n2 w0      -0.00054986  -0.00109817    3.33e-01
L0 n2 w1      -0.00109972  -0.00219633    3.33e-01
L0 n2 w2       0.00054986   0.00109817    3.33e-01
L0 n2 b       -0.00054986  -0.00109817    3.33e-01
L0 n3 w0      -0.98001650  -0.00569136    9.88e-01
L0 n3 w1      -1.96003299  -0.01138271    9.88e-01
L0 n3 w2       0.98001650   0.00569136    9.88e-01
L0 n3 b       -0.98001650  -0.00569136    9.88e-01
L1 n0 w0       0.45237212   0.45237212    4.32e-12
L1 n0 w1       0.42103985   0.42103985    6.14e-12
L1 n0 w2      -0.47005884  -0.4

## ReLU's kink

ReLU has no slope at exactly `z = 0`. If a nudge of `eps` pushes `z` across zero, the numerical slope averages the two sides and disagrees with backprop even though the code is right. Here the bias sits `1e-6` away from the kink, closer than `eps = 1e-5`.

In [5]:
kink = stackLayers([multipleNeuron(n=1, n_inputs=1, activation=relu, W=[[1.0]], b=[0.0])])
print_gradient_check(gradient_check(kink, [1e-6], [1.0], mean_squared_error))

param            analytic      numeric   rel error
L0 n0 w0      -0.00000200  -0.00000200    5.41e-07
L0 n0 b       -1.99999800  -1.09999395    2.90e-01
max relative error: 2.90e-01


## The tests

`tests/test_gradients.py` runs these checks on several networks (sigmoid, tanh, ReLU, both losses, up to three layers), checks the loss derivatives, and confirms a wrong derivative gets caught. Rerun it after any change to `nn/`.

In [6]:
!{sys.executable} -m tests.test_gradients

ok  test_single_neuron_matches_hand_example
ok  test_one_layer_sigmoid_mse
ok  test_two_layers_sigmoid_mse
ok  test_tanh_then_sigmoid_bce
ok  test_three_layers_mixed


ok  test_relu_hidden_layer
ok  test_tanh_then_linear_softmax_cross_entropy
ok  test_loss_derivatives_match_numerical
ok  test_wrong_derivative_is_caught
9 passed
